In [1]:
import pandas as pd
from pathlib import Path

In [2]:
ROOT = Path("E:/Data/lucia/logs/segmentation")
pattern = "*/*/*/*/csv/version_*/metrics.csv"
metrics_to_keep = {
    'testing/segments/mF1s@40-75-05', 'testing/segments/mF1s@50',
    'testing/segments/mF1b@05-25-05', 'testing/segments/mF1b@25',
    'testing/segments/mIoU',
    'testing/frames/macro_accuracy', 'testing/frames/balanced_accuracy'
}
files = list(ROOT.glob(pattern))
print(f"Found {len(files)} metric files")

Found 133 metric files


In [3]:
def create_results_df(files):
    rows = []
    for f in files:
        # Parse experiment info from the path
        # segmentation/<exp_id>/<dataset_id>/<exp_variant>/<seed>/csv/version_<i>/metrics.csv
        parts = f.parts
        exp_id, dataset_id, exp_variant, seed = parts[-7], parts[-6], parts[-5], parts[-4]
        version = parts[-2]  # "version_<i>"

        # Read CSV and keep the last row (test metrics)
        df = pd.read_csv(f)
        if df.empty:
            continue
        last_row = df.iloc[-1]

        # Keep only the metrics that exist in this file
        available = [m for m in df.columns if m in metrics_to_keep]
        metrics = last_row[available].to_dict()

        rows.append({
            "exp_id": exp_id,
            "dataset_id": dataset_id,
            "exp_variant": exp_variant,
            "seed": seed,
            "version": version,
            **metrics,
        })

    results = pd.DataFrame(rows)
    return results

results = create_results_df(files)
results.head()

,exp_id,dataset_id,exp_variant,seed,version,testing/frames/macro_accuracy,testing/segments/mF1b@05-25-05,testing/segments/mF1b@25,testing/segments/mF1s@40-75-05,testing/segments/mF1s@50,testing/segments/mIoU
0,1-comparison,dgs,bio,20899,version_0,0.354711,0.591022,0.669752,0.545613,0.652310,0.543798
1,1-comparison,dgs,bio_weights,194904,version_0,0.380319,0.556483,0.647788,0.474291,0.582649,0.511128
2,1-comparison,dgs,io,439217,version_0,0.816764,0.570083,0.658323,0.519511,0.623880,0.528092
3,1-comparison,dgs,io_weights,636180,version_0,0.839859,0.569662,0.667051,0.481961,0.580027,0.503557
4,1-comparison,dgs,off,122716,version_0,0.827750,0.573060,0.650167,0.593931,0.705737,0.590026


In [4]:
def create_excel_output(results):
    output_path = "experiments_metrics.xlsx"

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for exp_id, group in results.groupby("exp_id"):
            # Drop the exp_id column since it's now the sheet name
            # Sort for readability
            sheet_df = (group.drop(columns="exp_id")
                             .sort_values(["dataset_id", "exp_variant", "seed"]))

            # Excel sheet names: max 31 chars, no : \ / ? * [ ]
            sheet_name = str(exp_id)[:31].translate(str.maketrans("", "", r":\/?*[]"))

            sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"Wrote {output_path}")

create_excel_output(results)

Wrote experiments_metrics.xlsx


In [5]:
def output_csv_per_exp(results):
    output_dir = Path("metrics_by_experiment")
    output_dir.mkdir(exist_ok=True)

    for exp_id, group in results.groupby("exp_id"):
        sheet_df = (group.drop(columns="exp_id")
                         .sort_values(["dataset_id", "exp_variant", "seed"]))

        # Sanitize exp_id for use as a filename
        safe_name = "".join(c if c.isalnum() or c in "-_." else "_" for c in str(exp_id))
        out_path = output_dir / f"{safe_name}.csv"

        sheet_df.to_csv(out_path, index=False)
        print(f"Wrote {out_path} ({len(sheet_df)} rows)")

output_csv_per_exp(results)

Wrote metrics_by_experiment\1-comparison.csv (12 rows)
Wrote metrics_by_experiment\2-first-ablation.csv (6 rows)
Wrote metrics_by_experiment\3-mstcn-archs.csv (10 rows)
Wrote metrics_by_experiment\4-data-quantity.csv (10 rows)
Wrote metrics_by_experiment\5-mixing-sign-languages.csv (1 rows)
Wrote metrics_by_experiment\6-transformer-archs.csv (20 rows)
Wrote metrics_by_experiment\7-ms-transformer-archs.csv (10 rows)
Wrote metrics_by_experiment\8-mstcn-noise.csv (12 rows)
Wrote metrics_by_experiment\9-mstcn-kfolds.csv (16 rows)
Wrote metrics_by_experiment\comparison.csv (28 rows)
Wrote metrics_by_experiment\first-ablation.csv (8 rows)
